# 09. Xây dựng mô hình dự báo

Notebook này thực hiện huấn luyện và đánh giá các mô hình dự báo CPI nhóm Giao thông trên bộ dữ liệu đặc trưng đã xây dựng.

Do dữ liệu có tính chất chuỗi thời gian, tập huấn luyện và tập kiểm tra được chia theo thứ tự thời gian, không sử dụng phương pháp chia ngẫu nhiên.

In [1]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

df = pd.read_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/processed/feature_dataset.csv"
)

df["MonthYear"] = pd.to_datetime(
    df["MonthYear"]
).dt.to_period("M")

df.head()

,MonthYear,CPI,CPI_lag1,CPI_lag2,RON95_change_lag1,Diesel_change_lag1,Brent_change_lag1,USD_VND_change_lag1,RON95_MA3,Diesel_MA3,Brent_MA3,USDVND_MA3,Brent_WTI_Spread_lag1,Dummy_Tet,Dummy_Covid,Month_sin,Month_cos
0,2012-01,0.66,0.16,-0.01,0.000000,0.000000,-2.618037,0.050954,0.000000,-1.105845,-1.470473,0.299175,9.31,1,0,0.500000,8.660254e-01
1,2012-02,0.23,0.66,0.16,0.000000,0.000000,2.614258,0.069186,0.000000,0.000000,0.369956,0.184883,10.42,0,0,0.866025,5.000000e-01
2,2012-03,1.08,0.23,0.66,0.000000,0.000000,7.805583,0.000000,0.000000,0.000000,2.600601,0.040047,17.13,0,0,1.000000,6.123234e-17
3,2012-04,2.67,1.08,0.23,5.918899,3.953186,5.128635,0.000000,1.972966,1.317729,5.182825,0.023062,19.29,0,0,0.866025,-5.000000e-01
4,2012-05,1.32,2.67,1.08,1.341168,0.912694,-4.543643,0.000000,2.420022,1.621960,2.796858,0.000000,16.43,0,0,0.500000,-8.660254e-01


## 1. Chia tập Train/Test theo thời gian

Dữ liệu được chia theo thứ tự thời gian nhằm đảm bảo mô hình chỉ được huấn luyện bằng dữ liệu quá khứ và đánh giá trên các quan sát xảy ra sau đó.

- **Train:** 01/2012 – 12/2021
- **Test:** 01/2022 – 12/2024

Tập Test gồm 36 tháng cuối cùng và được sử dụng chung cho tất cả các mô hình nhằm đảm bảo việc so sánh kết quả được công bằng.

In [2]:
development_df = df[
    (df["MonthYear"] >= pd.Period("2012-01", freq="M")) &
    (df["MonthYear"] <= pd.Period("2021-12", freq="M"))
].copy()

test_df = df[
    (df["MonthYear"] >= pd.Period("2022-01", freq="M")) &
    (df["MonthYear"] <= pd.Period("2024-12", freq="M"))
].copy()

print("Development:")
print(
    development_df["MonthYear"].min(),
    "→",
    development_df["MonthYear"].max()
)
print("Số quan sát:", len(development_df))

print("\nTest:")
print(
    test_df["MonthYear"].min(),
    "→",
    test_df["MonthYear"].max()
)
print("Số quan sát:", len(test_df))

Development:
2012-01 → 2021-12
Số quan sát: 120

Test:
2022-01 → 2024-12
Số quan sát: 36


## 2. Thiết lập Expanding-window Validation

Trong tập Development 2012-2021, phương pháp Expanding-window Validation được sử dụng để lựa chọn cấu hình mô hình và siêu tham số.

Tập huấn luyện ban đầu gồm dữ liệu từ 01/2012 đến 12/2017. Sau mỗi lần dự báo một tháng validation, tập huấn luyện được mở rộng thêm quan sát mới và tiếp tục dự báo tháng kế tiếp.

Validation được thực hiện theo từng tháng từ 01/2018 đến 12/2021, tương ứng 48 fold. Cách thiết kế này đảm bảo tại mỗi thời điểm mô hình chỉ sử dụng dữ liệu đã xuất hiện trong quá khứ.

In [3]:
development_df = development_df.reset_index(drop=True)

validation_months = pd.period_range(
    start="2018-01",
    end="2021-12",
    freq="M"
)

expanding_splits = []

for val_month in validation_months:

    train_idx = development_df.index[
        development_df["MonthYear"] < val_month
    ].to_numpy()

    val_idx = development_df.index[
        development_df["MonthYear"] == val_month
    ].to_numpy()

    expanding_splits.append(
        (train_idx, val_idx)
    )

print("Số fold:", len(expanding_splits))

Số fold: 48


## 3. Mô hình Naive

Mô hình Naive được sử dụng làm mô hình cơ sở (baseline) để so sánh với các mô hình phức tạp hơn.

Mô hình giả định CPI của tháng tiếp theo bằng CPI của tháng gần nhất.

Mô hình được đánh giá trên cùng 48 fold của Expanding-window Validation trong giai đoạn Development.

In [4]:
naive_results = []

for fold, (train_idx, val_idx) in enumerate(expanding_splits, start=1):

    train_fold = development_df.iloc[train_idx]
    val_fold = development_df.iloc[val_idx]

    # CPI tháng cuối cùng trong tập train
    naive_pred = train_fold["CPI"].iloc[-1]

    # CPI thực tế của tháng validation
    actual = val_fold["CPI"].iloc[0]

    naive_results.append({
        "Fold": fold,
        "MonthYear": val_fold["MonthYear"].iloc[0],
        "Actual": actual,
        "Naive_Pred": naive_pred
    })

naive_val_results = pd.DataFrame(naive_results)

naive_val_results.head().round(2)

C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_20228\2536207439.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  naive_val_results.head().round(2)


,Fold,MonthYear,Actual,Naive_Pred
0,1,2018-01,1.17,0.84
1,2,2018-02,0.79,1.17
2,3,2018-03,-0.77,0.79
3,4,2018-04,1.18,-0.77
4,5,2018-05,1.72,1.18


In [5]:
print("Số dự báo:", len(naive_val_results))

print("\nFold đầu:")
display(naive_val_results.head(3).round(2))

print("\nFold cuối:")
display(naive_val_results.tail(3).round(2))

Số dự báo: 48

Fold đầu:


C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_20228\2175778500.py:4: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(naive_val_results.head(3).round(2))


,Fold,MonthYear,Actual,Naive_Pred
0,1,2018-01,1.17,0.84
1,2,2018-02,0.79,1.17
2,3,2018-03,-0.77,0.79



Fold cuối:


C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_20228\2175778500.py:7: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(naive_val_results.tail(3).round(2))


,Fold,MonthYear,Actual,Naive_Pred
45,46,2021-10,2.51,-0.16
46,47,2021-11,3.11,2.51
47,48,2021-12,-1.71,3.11


### Kết quả Validation của mô hình Naive

Mô hình Naive được đánh giá trên 48 tháng validation của Expanding-window. MAE và RMSE được sử dụng làm mốc tham chiếu để so sánh với các mô hình ElasticNet và ARIMAX ở các bước tiếp theo.

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

naive_val_mae = mean_absolute_error(
    naive_val_results["Actual"],
    naive_val_results["Naive_Pred"]
)

naive_val_rmse = np.sqrt(
    mean_squared_error(
        naive_val_results["Actual"],
        naive_val_results["Naive_Pred"]
    )
)

print(f"Naive Validation MAE : {naive_val_mae:.4f}")
print(f"Naive Validation RMSE: {naive_val_rmse:.4f}")

Naive Validation MAE : 2.0485
Naive Validation RMSE: 3.0927


## 4. Mô hình ElasticNet

ElasticNet được xây dựng trên các đặc trưng đã tạo ở bước Feature Engineering.

Các siêu tham số `alpha` và `l1_ratio` được lựa chọn bằng Expanding-window Validation trên giai đoạn 2012–2021. Trong mỗi fold, bộ chuẩn hóa dữ liệu chỉ được fit trên tập Train và sau đó áp dụng cho tập Validation nhằm tránh rò rỉ thông tin tương lai.

In [7]:
feature_cols = [
    "CPI_lag1",
    "CPI_lag2",

    "RON95_change_lag1",
    "Diesel_change_lag1",
    "Brent_change_lag1",
    "USD_VND_change_lag1",

    "RON95_MA3",
    "Diesel_MA3",
    "Brent_MA3",
    "USDVND_MA3",

    "Brent_WTI_Spread_lag1",

    "Dummy_Tet",
    "Dummy_Covid",

    "Month_sin",
    "Month_cos"
]

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

alpha_grid = [0.001, 0.01, 0.1, 1, 10]
l1_ratio_grid = [0.1, 0.3, 0.5, 0.7, 0.9]

elastic_results = []

for alpha in alpha_grid:
    for l1_ratio in l1_ratio_grid:

        actual_values = []
        pred_values = []

        for train_idx, val_idx in expanding_splits:

            train_fold = development_df.iloc[train_idx]
            val_fold = development_df.iloc[val_idx]

            X_train_fold = train_fold[feature_cols]
            y_train_fold = train_fold["CPI"]

            X_val_fold = val_fold[feature_cols]
            y_val_fold = val_fold["CPI"]

            # Scale chỉ dựa trên Train của từng fold
            scaler = StandardScaler()

            X_train_scaled = scaler.fit_transform(X_train_fold)
            X_val_scaled = scaler.transform(X_val_fold)

            model = ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=10000
            )

            model.fit(
                X_train_scaled,
                y_train_fold
            )

            pred = model.predict(X_val_scaled)[0]

            actual_values.append(y_val_fold.iloc[0])
            pred_values.append(pred)

        rmse = np.sqrt(
            mean_squared_error(
                actual_values,
                pred_values
            )
        )

        elastic_results.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "RMSE": rmse
        })

elastic_search = (
    pd.DataFrame(elastic_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

elastic_search.head(10).round(4)

,alpha,l1_ratio,RMSE
0,0.1,0.9,2.1856
1,0.1,0.7,2.1991
2,1.0,0.5,2.2173
3,0.1,0.5,2.2197
4,1.0,0.1,2.2256
5,1.0,0.7,2.2272
6,1.0,0.3,2.2321
7,0.1,0.3,2.2360
8,0.1,0.1,2.2526
9,1.0,0.9,2.2789


### Nhận xét

Kết quả Expanding-window Validation cho thấy cấu hình ElasticNet tốt nhất có `alpha = 0,1` và `l1_ratio = 0,9`, với **RMSE = 2,1856**.

Kết quả này thấp hơn RMSE của mô hình Naive (**3,0927**), cho thấy việc sử dụng các đặc trưng về độ trễ, biến động giá, xu hướng ngắn hạn và mùa vụ giúp cải thiện khả năng dự báo CPI.

Cấu hình trên được giữ lại làm cấu hình ElasticNet ứng viên cho các bước tiếp theo.

In [9]:
best_alpha = 0.1
best_l1_ratio = 0.9

elastic_val_results = []

for fold, (train_idx, val_idx) in enumerate(expanding_splits, start=1):

    train_fold = development_df.iloc[train_idx]
    val_fold = development_df.iloc[val_idx]

    X_train_fold = train_fold[feature_cols]
    y_train_fold = train_fold["CPI"]

    X_val_fold = val_fold[feature_cols]
    y_val_fold = val_fold["CPI"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_val_scaled = scaler.transform(X_val_fold)

    model = ElasticNet(
        alpha=best_alpha,
        l1_ratio=best_l1_ratio,
        max_iter=10000
    )

    model.fit(X_train_scaled, y_train_fold)

    pred = model.predict(X_val_scaled)[0]

    elastic_val_results.append({
        "Fold": fold,
        "MonthYear": val_fold["MonthYear"].iloc[0],
        "Actual": y_val_fold.iloc[0],
        "ElasticNet_Pred": pred
    })

elastic_val_results = pd.DataFrame(elastic_val_results)

elastic_val_results.head().round(2)

C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_20228\4020073751.py:41: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  elastic_val_results.head().round(2)


,Fold,MonthYear,Actual,ElasticNet_Pred
0,1,2018-01,1.17,0.64
1,2,2018-02,0.79,1.58
2,3,2018-03,-0.77,-0.43
3,4,2018-04,1.18,-0.01
4,5,2018-05,1.72,1.35


## 5. Mô hình ARIMAX

ARIMAX mở rộng mô hình ARIMA bằng cách bổ sung các biến ngoại sinh nhằm khai thác thêm thông tin từ biến động giá nhiên liệu, giá dầu thế giới, tỷ giá và các yếu tố sự kiện.

Cấu hình ARIMAX được lựa chọn trong giai đoạn Development bằng Expanding-window Validation, tương tự ElasticNet.

In [10]:
arimax_exog_cols = [
    "RON95_change_lag1",
    "Diesel_change_lag1",
    "Brent_change_lag1",
    "USD_VND_change_lag1",

    "Dummy_Tet",
    "Dummy_Covid",

    "Month_sin",
    "Month_cos"
]

### 6.1. Lựa chọn biến ngoại sinh

Các bộ biến ngoại sinh được xây dựng dựa trên kết quả EDA và Feature Engineering. Các bộ ứng viên được so sánh bằng AIC và BIC trên tập Development.

Bộ biến có AIC/BIC thấp hơn được xem là phù hợp hơn và được sử dụng cho bước xác định cấu hình ARIMAX tiếp theo.

In [11]:
arimax_exog_sets = {
    "Set_1": [
        "RON95_change_lag1",
        "Diesel_change_lag1",
        "Brent_change_lag1",
        "USD_VND_change_lag1"
    ],

    "Set_2": [
        "RON95_change_lag1",
        "Diesel_change_lag1",
        "Brent_change_lag1",
        "USD_VND_change_lag1",
        "Dummy_Tet",
        "Dummy_Covid"
    ],

    "Set_3": [
        "RON95_change_lag1",
        "Diesel_change_lag1",
        "Brent_change_lag1",
        "USD_VND_change_lag1",
        "Dummy_Tet",
        "Dummy_Covid",
        "Month_sin",
        "Month_cos"
    ],

    "Set_4": [
        "RON95_change_lag1",
        "Diesel_change_lag1",
        "Brent_change_lag1",
        "USD_VND_change_lag1",
        "RON95_MA3",
        "Diesel_MA3",
        "Brent_MA3",
        "USDVND_MA3",
        "Dummy_Tet",
        "Dummy_Covid",
        "Month_sin",
        "Month_cos"
    ]
}

In [12]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

exog_compare_results = []

for set_name, cols in arimax_exog_sets.items():

    model = SARIMAX(
        development_df["CPI"],
        exog=development_df[cols],
        order=(2, 0, 0),
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted = model.fit(disp=False)

    exog_compare_results.append({
        "Exog_Set": set_name,
        "Num_Features": len(cols),
        "AIC": fitted.aic,
        "BIC": fitted.bic
    })

exog_compare = (
    pd.DataFrame(exog_compare_results)
    .sort_values("AIC")
    .reset_index(drop=True)
)

exog_compare.round(2)

d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,Exog_Set,Num_Features,AIC,BIC
0,Set_1,4,425.66,447.82
1,Set_2,6,427.52,455.22
2,Set_3,8,429.87,463.12
3,Set_4,12,434.93,479.26


### Nhận xét

Kết quả so sánh cho thấy **Set_1** có AIC và BIC thấp nhất, lần lượt là **425,66** và **447,82**. Việc bổ sung thêm các biến Dummy, mùa vụ và MA3 không làm cải thiện AIC/BIC.

Do đó, bộ biến ngoại sinh được lựa chọn cho ARIMAX gồm:

- `RON95_change_lag1`
- `Diesel_change_lag1`
- `Brent_change_lag1`
- `USD_VND_change_lag1`

Bộ biến này được sử dụng để tiếp tục xác định cấu hình `(p,d,q)` của mô hình ARIMAX.

### 6.2. Lựa chọn tham số ARIMAX

Sau khi xác định bộ biến ngoại sinh, các tổ hợp `(p,d,q)` được thử nghiệm trên tập Development. Do chuỗi CPI đã dừng theo kiểm định ADF nên `d = 0`.

Các mô hình ứng viên được so sánh dựa trên AIC và BIC để lựa chọn cấu hình phù hợp.

In [13]:
best_exog_cols = [
    "RON95_change_lag1",
    "Diesel_change_lag1",
    "Brent_change_lag1",
    "USD_VND_change_lag1"
]

arimax_order_results = []

for p in range(0, 5):
    for q in range(0, 5):

        try:
            model = SARIMAX(
                development_df["CPI"],
                exog=development_df[best_exog_cols],
                order=(p, 0, q),
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted = model.fit(disp=False)

            arimax_order_results.append({
                "p": p,
                "d": 0,
                "q": q,
                "AIC": fitted.aic,
                "BIC": fitted.bic
            })

        except:
            continue

arimax_order_search = (
    pd.DataFrame(arimax_order_results)
    .sort_values("AIC")
    .reset_index(drop=True)
)

arimax_order_search.head(10).round(2)

d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to conve

,p,d,q,AIC,BIC
0,0,0,4,417.95,445.40
1,1,0,4,419.86,450.06
2,0,0,3,420.23,445.01
3,1,0,2,420.41,445.27
4,1,0,3,421.15,448.69
5,3,0,1,421.69,449.31
6,2,0,4,421.85,454.79
7,3,0,4,421.95,457.64
8,4,0,0,422.73,450.26
9,4,0,1,422.98,453.27


### Nhận xét

Kết quả tìm kiếm cho thấy **ARIMAX(0,0,4)** có AIC thấp nhất (**417,95**), trong khi **ARIMAX(0,0,3)** có BIC thấp nhất (**445,01**).

Do AIC và BIC đưa ra lựa chọn khác nhau và mức chênh lệch giữa các mô hình khá nhỏ, các cấu hình tốt nhất được tiếp tục đánh giá bằng **Expanding-window Validation**. Cấu hình có RMSE validation thấp nhất sẽ được lựa chọn.

In [14]:
candidate_orders = [
    (0, 0, 4),
    (0, 0, 3),
    (1, 0, 2)
]

In [15]:
arimax_validation_results = []

for order in candidate_orders:

    actual_values = []
    pred_values = []

    for train_idx, val_idx in expanding_splits:

        train_fold = development_df.iloc[train_idx]
        val_fold = development_df.iloc[val_idx]

        model = SARIMAX(
            train_fold["CPI"],
            exog=train_fold[best_exog_cols],
            order=order,
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted = model.fit(disp=False)

        pred = fitted.forecast(
            steps=1,
            exog=val_fold[best_exog_cols]
        ).iloc[0]

        actual_values.append(
            val_fold["CPI"].iloc[0]
        )

        pred_values.append(pred)

    rmse = np.sqrt(
        mean_squared_error(
            actual_values,
            pred_values
        )
    )

    mae = mean_absolute_error(
        actual_values,
        pred_values
    )

    arimax_validation_results.append({
        "order": str(order),
        "MAE": mae,
        "RMSE": rmse
    })

arimax_validation_compare = (
    pd.DataFrame(arimax_validation_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

arimax_validation_compare.round(4)

d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to conve

,order,MAE,RMSE
0,"(1, 0, 2)",1.4117,2.2829
1,"(0, 0, 3)",1.4522,2.3050
2,"(0, 0, 4)",1.4647,2.3308


### Nhận xét

Kết quả Expanding-window Validation cho thấy **ARIMAX(1,0,2)** đạt kết quả tốt nhất trong các cấu hình ứng viên, với **MAE = 1,4117** và **RMSE = 2,2829**.

Mặc dù các cấu hình khác có AIC hoặc BIC thấp hơn trên toàn bộ tập Development, ARIMAX(1,0,2) cho sai số dự báo thấp nhất trên các fold validation. Do đó, cấu hình này được lựa chọn làm mô hình ARIMAX cuối cùng.

So với mô hình Naive (**RMSE = 3,0927**), ARIMAX cải thiện đáng kể khả năng dự báo. Tuy nhiên, ElasticNet hiện vẫn có RMSE thấp hơn (**2,1856**).

In [16]:
best_arimax_order = (1, 0, 2)

arimax_val_results = []

for fold, (train_idx, val_idx) in enumerate(
    expanding_splits,
    start=1
):

    train_fold = development_df.iloc[train_idx]
    val_fold = development_df.iloc[val_idx]

    model = SARIMAX(
        train_fold["CPI"],
        exog=train_fold[best_exog_cols],
        order=best_arimax_order,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted = model.fit(disp=False)

    pred = fitted.forecast(
        steps=1,
        exog=val_fold[best_exog_cols]
    ).iloc[0]

    arimax_val_results.append({
        "Fold": fold,
        "MonthYear": val_fold["MonthYear"].iloc[0],
        "Actual": val_fold["CPI"].iloc[0],
        "ARIMAX_Pred": pred
    })

arimax_val_results = pd.DataFrame(
    arimax_val_results
)

arimax_val_results.head().round(2)

d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to conve

,Fold,MonthYear,Actual,ARIMAX_Pred
0,1,2018-01,1.17,0.34
1,2,2018-02,0.79,1.56
2,3,2018-03,-0.77,-0.74
3,4,2018-04,1.18,0.10
4,5,2018-05,1.72,1.34


## 6. Mô hình Ensemble

Sau khi xác định cấu hình phù hợp cho ba mô hình đơn gồm Naive, ElasticNet và ARIMAX, các mô hình được kết hợp theo phương pháp Weighted Averaging nhằm đánh giá khả năng cải thiện kết quả dự báo khi tổng hợp thông tin từ nhiều mô hình.

Ba cặp mô hình Ensemble được xem xét gồm:

- Naive + ElasticNet,
- Naive + ARIMAX,
- ElasticNet + ARIMAX

Đối với mỗi cặp, trọng số của hai mô hình thành phần được xác định dựa trên nghịch đảo RMSE thu được từ Expanding-window Validation. Mô hình có RMSE Validation thấp hơn sẽ nhận trọng số lớn hơn.

In [17]:
# Tổng hợp RMSE Validation của 3 mô hình đơn

validation_rmse = {
    "Naive": naive_val_rmse,
    "ElasticNet": elastic_search.iloc[0]["RMSE"],
    "ARIMAX": arimax_validation_compare.iloc[0]["RMSE"],
}

validation_rmse_df = pd.DataFrame([
    {
        "Model": model,
        "Validation_RMSE": rmse
    }
    for model, rmse in validation_rmse.items()
])

validation_rmse_df = (
    validation_rmse_df
    .sort_values("Validation_RMSE")
    .reset_index(drop=True)
)

validation_rmse_df.round(4)

,Model,Validation_RMSE
0,ElasticNet,2.1856
1,ARIMAX,2.2829
2,Naive,3.0927


In [18]:
validation_predictions = {
    
    "Naive": (
        naive_val_results[
            ["Fold", "MonthYear", "Actual", "Naive_Pred"]
        ]
        .rename(
            columns={
                "Naive_Pred": "Prediction"
            }
        )
        .copy()
    ),

    "ElasticNet": (
        elastic_val_results[
            ["Fold", "MonthYear", "Actual", "ElasticNet_Pred"]
        ]
        .rename(
            columns={
                "ElasticNet_Pred": "Prediction"
            }
        )
        .copy()
    ),

    "ARIMAX": (
        arimax_val_results[
            ["Fold", "MonthYear", "Actual", "ARIMAX_Pred"]
        ]
        .rename(
            columns={
                "ARIMAX_Pred": "Prediction"
            }
        )
        .copy()
    ),
}

for model_name, pred_df in validation_predictions.items():
    
    print(f"\n{model_name}")
    
    display(
        pred_df.head(3).round(2)
    )

C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_20228\146986310.py:45: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  pred_df.head(3).round(2)



Naive


,Fold,MonthYear,Actual,Prediction
0,1,2018-01,1.17,0.84
1,2,2018-02,0.79,1.17
2,3,2018-03,-0.77,0.79



ElasticNet


C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_20228\146986310.py:45: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  pred_df.head(3).round(2)


,Fold,MonthYear,Actual,Prediction
0,1,2018-01,1.17,0.64
1,2,2018-02,0.79,1.58
2,3,2018-03,-0.77,-0.43



ARIMAX


C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_20228\146986310.py:45: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  pred_df.head(3).round(2)


,Fold,MonthYear,Actual,Prediction
0,1,2018-01,1.17,0.34
1,2,2018-02,0.79,1.56
2,3,2018-03,-0.77,-0.74


In [19]:
# Xác định trọng số cho từng cặp Ensemble

def calculate_inverse_rmse_weights(
    model_1,
    model_2,
    rmse_dict,
):
    rmse_1 = rmse_dict[model_1]
    rmse_2 = rmse_dict[model_2]

    inv_1 = 1 / rmse_1
    inv_2 = 1 / rmse_2

    weight_1 = inv_1 / (inv_1 + inv_2)
    weight_2 = inv_2 / (inv_1 + inv_2)

    return weight_1, weight_2

ensemble_pairs = [
    ("Naive", "ElasticNet"),
    ("Naive", "ARIMAX"),
    ("ElasticNet", "ARIMAX"),
]

ensemble_weights = {}

for model_1, model_2 in ensemble_pairs:

    weight_1, weight_2 = calculate_inverse_rmse_weights(
        model_1,
        model_2,
        validation_rmse,
    )

    pair_name = f"{model_1} + {model_2}"

    ensemble_weights[pair_name] = {
        "Model_1": model_1,
        "Model_2": model_2,
        "Weight_1": weight_1,
        "Weight_2": weight_2,
    }

ensemble_weights_df = pd.DataFrame([
    {
        "Pair": pair_name,
        "Model_1": info["Model_1"],
        "Weight_1": info["Weight_1"],
        "Model_2": info["Model_2"],
        "Weight_2": info["Weight_2"],
    }
    for pair_name, info in ensemble_weights.items()
])

ensemble_weights_df.round(4)

,Pair,Model_1,Weight_1,Model_2,Weight_2
0,Naive + ElasticNet,Naive,0.4141,ElasticNet,0.5859
1,Naive + ARIMAX,Naive,0.4247,ARIMAX,0.5753
2,ElasticNet + ARIMAX,ElasticNet,0.5109,ARIMAX,0.4891


In [20]:
ensemble_weights_df["Weight_1"] + ensemble_weights_df["Weight_2"] == 1

0    True
1    True
2    True
dtype: bool

In [21]:
def build_ensemble_validation(
    model_1,
    model_2,
    weight_1,
    weight_2,
    prediction_dict,
):

    pred_1 = (
        prediction_dict[model_1]
        .copy()
        .rename(
            columns={
                "Prediction": f"{model_1}_Pred"
            }
        )
    )

    pred_2 = (
        prediction_dict[model_2][
            ["Fold", "MonthYear", "Prediction"]
        ]
        .copy()
        .rename(
            columns={
                "Prediction": f"{model_2}_Pred"
            }
        )
    )

    # Ghép kết quả dự báo của 2 mô hình
    ensemble_df = pred_1.merge(
        pred_2,
        on=["Fold", "MonthYear"],
        how="inner",
    )

    # Tính dự báo Ensemble
    ensemble_df["Ensemble_Pred"] = (
        weight_1 * ensemble_df[f"{model_1}_Pred"]
        +
        weight_2 * ensemble_df[f"{model_2}_Pred"]
    )

    # MAE
    mae = mean_absolute_error(
        ensemble_df["Actual"],
        ensemble_df["Ensemble_Pred"],
    )

    # RMSE
    rmse = np.sqrt(
        mean_squared_error(
            ensemble_df["Actual"],
            ensemble_df["Ensemble_Pred"],
        )
    )

    return ensemble_df, mae, rmse

In [22]:
ensemble_validation_results = {}

for pair_name, info in ensemble_weights.items():

    model_1 = info["Model_1"]
    model_2 = info["Model_2"]

    weight_1 = info["Weight_1"]
    weight_2 = info["Weight_2"]

    pred_df, mae, rmse = build_ensemble_validation(
        model_1=model_1,
        model_2=model_2,
        weight_1=weight_1,
        weight_2=weight_2,
        prediction_dict=validation_predictions,
    )

    ensemble_validation_results[pair_name] = {
        "Model_1": model_1,
        "Model_2": model_2,
        "Weight_1": weight_1,
        "Weight_2": weight_2,
        "MAE": mae,
        "RMSE": rmse,
        "Predictions": pred_df,
    }

ensemble_compare = pd.DataFrame([
    {
        "Pair": pair_name,
        "Weight_1": result["Weight_1"],
        "Weight_2": result["Weight_2"],
        "MAE": result["MAE"],
        "RMSE": result["RMSE"],
    }
    for pair_name, result in ensemble_validation_results.items()
])

ensemble_compare = (
    ensemble_compare
    .sort_values("RMSE")
    .reset_index(drop=True)
)

ensemble_compare.round(4)

,Pair,Weight_1,Weight_2,MAE,RMSE
0,ElasticNet + ARIMAX,0.5109,0.4891,1.3555,2.1935
1,Naive + ElasticNet,0.4141,0.5859,1.5707,2.4224
2,Naive + ARIMAX,0.4247,0.5753,1.5464,2.4228


In [25]:
best_ensemble_row = ensemble_compare.iloc[0]

best_ensemble_pair = best_ensemble_row["Pair"]

best_ensemble_rmse = best_ensemble_row["RMSE"]

best_ensemble_mae = best_ensemble_row["MAE"]

best_ensemble_weight_1 = best_ensemble_row["Weight_1"]

best_ensemble_weight_2 = best_ensemble_row["Weight_2"]


print(
    f"Cặp Ensemble tốt nhất: {best_ensemble_pair}"
)

print(
    f"Validation MAE: {best_ensemble_mae:.4f}"
)

print(
    f"Validation RMSE: {best_ensemble_rmse:.4f}"
)

print(
    f"Weight 1: {best_ensemble_weight_1:.4f}"
)

print(
    f"Weight 2: {best_ensemble_weight_2:.4f}"
)

validation_model_predictions = (
    validation_predictions["Naive"][
        [
            "Fold",
            "MonthYear",
            "Actual",
            "Prediction",
        ]
    ]
    .copy()
    .rename(
        columns={
            "Prediction": "Naive_Pred"
        }
    )
)


# ElasticNet
validation_model_predictions["ElasticNet_Pred"] = (
    validation_predictions[
        "ElasticNet"
    ]["Prediction"].values
)


# ARIMAX
validation_model_predictions["ARIMAX_Pred"] = (
    validation_predictions[
        "ARIMAX"
    ]["Prediction"].values
)


# 3 Ensemble
for pair_name, result in ensemble_validation_results.items():

    pred_col = (
        pair_name
        .replace(" + ", "_")
        + "_Pred"
    )

    validation_model_predictions[pred_col] = (
        result["Predictions"][
            "Ensemble_Pred"
        ].values
    )


validation_model_predictions.head().round(2)

Cặp Ensemble tốt nhất: ElasticNet + ARIMAX
Validation MAE: 1.3555
Validation RMSE: 2.1935
Weight 1: 0.5109
Weight 2: 0.4891


C:\Users\Phuong Uyen\AppData\Local\Temp\ipykernel_20228\1279438757.py:84: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  validation_model_predictions.head().round(2)


,Fold,MonthYear,Actual,Naive_Pred,ElasticNet_Pred,ARIMAX_Pred,Naive_ElasticNet_Pred,Naive_ARIMAX_Pred,ElasticNet_ARIMAX_Pred
0,1,2018-01,1.17,0.84,0.64,0.34,0.73,0.55,0.50
1,2,2018-02,0.79,1.17,1.58,1.56,1.41,1.40,1.57
2,3,2018-03,-0.77,0.79,-0.43,-0.74,0.08,-0.09,-0.58
3,4,2018-04,1.18,-0.77,-0.01,0.10,-0.33,-0.27,0.04
4,5,2018-05,1.72,1.18,1.35,1.34,1.28,1.27,1.34


### Nhận xét

Cặp Ensemble có **RMSE Validation thấp nhất** được chọn làm cấu hình Ensemble đại diện. Cấu hình và trọng số này được giữ cố định để tiếp tục đánh giá trên tập Test 2022–2024.


In [27]:
validation_model_compare = pd.DataFrame([
    {
        "Model": "Naive",
        "RMSE": validation_rmse["Naive"],
    },
    {
        "Model": "ElasticNet",
        "RMSE": validation_rmse["ElasticNet"],
    },
    {
        "Model": "ARIMAX",
        "RMSE": validation_rmse["ARIMAX"],
    },
])

validation_model_compare = pd.concat(
    [
        validation_model_compare,

        ensemble_compare[
            ["Pair", "RMSE"]
        ].rename(
            columns={
                "Pair": "Model"
            }
        ),
    ],
    ignore_index=True,
)

validation_model_compare = (
    validation_model_compare
    .sort_values("RMSE")
    .reset_index(drop=True)
)

validation_model_compare.round(4)

,Model,RMSE
0,ElasticNet,2.1856
1,ElasticNet + ARIMAX,2.1935
2,ARIMAX,2.2829
3,Naive + ElasticNet,2.4224
4,Naive + ARIMAX,2.4228
5,Naive,3.0927


### Nhận xét

ElasticNet đạt RMSE Validation thấp nhất (**2.1856**) và được xác định là mô hình tốt nhất trên Validation. Trong các mô hình kết hợp, **ElasticNet + ARIMAX** đạt kết quả tốt nhất với RMSE **2.1935** và được chọn làm cấu hình Ensemble đại diện để đánh giá trên tập Test.


In [28]:
validation_model_compare.to_csv(
    "../data/processed/model_validation_metrics.csv",
    index=False
)

## 8. Huấn luyện mô hình cuối trên toàn bộ Development

Sau khi hoàn tất Expanding-window Validation, các cấu hình tốt nhất được sử dụng để huấn luyện lại mô hình trên toàn bộ tập Development từ 01/2012 đến 12/2021.

### 8.1. ElasticNet

ElasticNet sử dụng cấu hình được lựa chọn từ Validation:

- `alpha = 0.1`
- `l1_ratio = 0.9`

Bộ chuẩn hóa `StandardScaler` được fit chỉ trên tập Development và được giữ lại để áp dụng cho dữ liệu Test.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

# X và y của toàn bộ Development
X_dev = development_df[feature_cols].copy()
y_dev = development_df["CPI"].copy()

# Chuẩn hóa trên Development
elastic_scaler = StandardScaler()

X_dev_scaled = elastic_scaler.fit_transform(X_dev)

# Fit ElasticNet cuối cùng
elastic_final = ElasticNet(
    alpha=0.1,
    l1_ratio=0.9,
    max_iter=10000
)

elastic_final.fit(
    X_dev_scaled,
    y_dev
)

print("ElasticNet final đã được huấn luyện.")
print("Số quan sát Development:", len(X_dev))
print("Số feature:", len(feature_cols))

ElasticNet final đã được huấn luyện.
Số quan sát Development: 120
Số feature: 15


### 8.2. Huấn luyện lại ARIMAX trên toàn bộ Development

Sau Expanding-window Validation, cấu hình ARIMAX được lựa chọn là `ARIMAX(1,0,2)` với bộ biến ngoại sinh gồm biến động RON95, Diesel, Brent và USD/VND ở độ trễ 1 tháng.

Mô hình được huấn luyện lại trên toàn bộ tập Development từ 01/2012 đến 12/2021 để chuẩn bị cho bước dự báo trên tập Test.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

best_arimax_order = (1, 0, 2)

best_exog_cols = [
    "RON95_change_lag1",
    "Diesel_change_lag1",
    "Brent_change_lag1",
    "USD_VND_change_lag1"
]

arimax_final = SARIMAX(
    development_df["CPI"],
    exog=development_df[best_exog_cols],
    order=best_arimax_order,
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False
)

arimax_final_fit = arimax_final.fit(
    disp=False
)

print("ARIMAX đã được huấn luyện lại trên toàn bộ Development.")
print("Số quan sát Development:", len(development_df))
print("Cấu hình:", best_arimax_order)
print("Số biến ngoại sinh:", len(best_exog_cols))

ARIMAX đã được huấn luyện lại trên toàn bộ Development.
Số quan sát Development: 120
Cấu hình: (1, 0, 2)
Số biến ngoại sinh: 4


d:\HK3 - năm 3\Đồ Án\Code\vietnam-transportation-cpi-forecasting\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 9. Dự báo trên tập Test 2022–2024

Sau khi các mô hình được lựa chọn và huấn luyện lại trên toàn bộ tập Development 2012–2021, các mô hình được sử dụng để dự báo CPI trên tập Test từ 01/2022 đến 12/2024.

Tập Test chỉ được sử dụng để đánh giá cuối cùng và không tham gia vào quá trình lựa chọn cấu hình hoặc điều chỉnh siêu tham số.

### 9.1. Dự báo bằng Naive và ElasticNet

In [ ]:
# Sắp xếp Test theo thời gian
test_df = test_df.sort_values("MonthYear").reset_index(drop=True)

# Bảng lưu kết quả dự báo
test_predictions = pd.DataFrame({
    "MonthYear": test_df["MonthYear"].astype(str),
    "Actual": test_df["CPI"].values
})

# -------------------------
# Naive
# CPI dự báo = CPI tháng trước
# -------------------------
test_predictions["Naive_Pred"] = (
    test_df["CPI_lag1"].values
)

# -------------------------
# ElasticNet
# -------------------------
X_test = test_df[feature_cols].copy()

# Chỉ transform bằng scaler đã fit trên Development
X_test_scaled = elastic_scaler.transform(X_test)

test_predictions["ElasticNet_Pred"] = (
    elastic_final.predict(X_test_scaled)
)

test_predictions.head().round(2)

,MonthYear,Actual,Naive_Pred,ElasticNet_Pred
0,2022-01,1.18,-1.71,-1.45
1,2022-02,2.35,1.18,2.13
2,2022-03,4.80,2.35,1.95
3,2022-04,-0.59,4.80,3.66
4,2022-05,2.34,-0.59,-1.39


### 9.2. Dự báo bằng ARIMAX

ARIMAX sử dụng cấu hình `ARIMAX(1,0,2)` đã được lựa chọn trong giai đoạn Development.

Trong giai đoạn Test, các tham số của mô hình được giữ cố định. Sau mỗi tháng, giá trị CPI thực tế đã quan sát được cập nhật vào trạng thái của mô hình để thực hiện dự báo một bước cho tháng tiếp theo. Mô hình không được huấn luyện hoặc điều chỉnh lại bằng dữ liệu Test.

In [ ]:
# Tạo bản sao riêng cho ARIMAX
arimax_test_df = test_df.copy()

# Cho index Test nối tiếp Development
arimax_test_df.index = pd.RangeIndex(
    start=len(development_df),
    stop=len(development_df) + len(arimax_test_df)
)

arimax_test_pred = []

# Model đã fit trên Development 2012–2021
current_fit = arimax_final_fit

for i in range(len(arimax_test_df)):

    current_idx = arimax_test_df.index[i]

    # Biến ngoại sinh của tháng cần dự báo
    exog_next = arimax_test_df.loc[
        [current_idx],
        best_exog_cols
    ]

    # Dự báo 1 tháng
    forecast = current_fit.forecast(
        steps=1,
        exog=exog_next
    )

    arimax_test_pred.append(
        forecast.iloc[0]
    )

    # CPI thực tế của tháng vừa dự báo
    actual = arimax_test_df.loc[
        current_idx,
        "CPI"
    ]

    # Tạo Series có index nối tiếp model
    actual_series = pd.Series(
        [actual],
        index=[current_idx],
        name="CPI"
    )

    # Cập nhật quan sát mới, KHÔNG fit lại tham số
    current_fit = current_fit.append(
        endog=actual_series,
        exog=exog_next,
        refit=False
    )

# Ghép dự báo vào bảng kết quả
test_predictions["ARIMAX_Pred"] = arimax_test_pred

test_predictions.head().round(2)

,MonthYear,Actual,Naive_Pred,ElasticNet_Pred,ARIMAX_Pred
0,2022-01,1.18,-1.71,-1.45,-1.56
1,2022-02,2.35,1.18,2.13,1.35
2,2022-03,4.80,2.35,1.95,1.45
3,2022-04,-0.59,4.80,3.66,2.73
4,2022-05,2.34,-0.59,-1.39,-2.35


In [ ]:
print(development_df.index.min(), development_df.index.max())
print(arimax_test_df.index.min(), arimax_test_df.index.max())

0 119
120 155


### 9.3. Dự báo bằng mô hình Ensemble

Mô hình Ensemble kết hợp dự báo của ElasticNet và ARIMAX bằng các trọng số đã được xác định từ RMSE trong giai đoạn Expanding-window Validation.

Các trọng số được giữ cố định khi dự báo trên tập Test nhằm đảm bảo dữ liệu Test không tham gia vào quá trình lựa chọn hoặc điều chỉnh mô hình.

In [ ]:
info_ne = ensemble_validation_results[
    "Naive + ElasticNet"
]

w_naive_ne = info_ne["Weight_1"]
w_elastic_ne = info_ne["Weight_2"]

test_predictions["Naive_ElasticNet_Pred"] = (
    w_naive_ne
    * test_predictions["Naive_Pred"]
    +
    w_elastic_ne
    * test_predictions["ElasticNet_Pred"]
)

info_na = ensemble_validation_results[
    "Naive + ARIMAX"
]

w_naive_na = info_na["Weight_1"]
w_arimax_na = info_na["Weight_2"]

test_predictions["Naive_ARIMAX_Pred"] = (
    w_naive_na
    * test_predictions["Naive_Pred"]
    +
    w_arimax_na
    * test_predictions["ARIMAX_Pred"]
)

info_ea = ensemble_validation_results[
    "ElasticNet + ARIMAX"
]

w_elastic_ea = info_ea["Weight_1"]
w_arimax_ea = info_ea["Weight_2"]

test_predictions["ElasticNet_ARIMAX_Pred"] = (
    w_elastic_ea
    * test_predictions["ElasticNet_Pred"]
    +
    w_arimax_ea
    * test_predictions["ARIMAX_Pred"]
)

# Xóa cột Ensemble cũ tồn tại
test_predictions = test_predictions.drop(
    columns=["Ensemble_Pred"],
    errors="ignore"
)

test_predictions.head().round(2)

,MonthYear,Actual,Naive_Pred,ElasticNet_Pred,ARIMAX_Pred,Naive_ElasticNet_Pred,Naive_ARIMAX_Pred,ElasticNet_ARIMAX_Pred
0,2022-01,1.18,-1.71,-1.45,-1.56,-1.56,-1.62,-1.51
1,2022-02,2.35,1.18,2.13,1.35,1.74,1.28,1.75
2,2022-03,4.80,2.35,1.95,1.45,2.11,1.83,1.70
3,2022-04,-0.59,4.80,3.66,2.73,4.13,3.61,3.21
4,2022-05,2.34,-0.59,-1.39,-2.35,-1.06,-1.60,-1.86


In [ ]:
print("Số tháng Test:", len(test_predictions))
print("Số giá trị thiếu:")
print(test_predictions.isna().sum())

Số tháng Test: 36
Số giá trị thiếu:
MonthYear                 0
Actual                    0
Naive_Pred                0
ElasticNet_Pred           0
ARIMAX_Pred               0
Naive_ElasticNet_Pred     0
Naive_ARIMAX_Pred         0
ElasticNet_ARIMAX_Pred    0
dtype: int64


## 10. Phân tích cú sốc trong giai đoạn diễn ra xung đột Nga-Ukraine

In [ ]:
shock_df = test_predictions.copy()

shock_df["MonthYear"] = pd.to_datetime(
    shock_df["MonthYear"].astype(str),
    errors="coerce"
)

# Sai số tuyệt đối của 2 mô hình tốt nhất
shock_df["Abs_Error_ElasticNet"] = (
    shock_df["Actual"]
    - shock_df["ElasticNet_Pred"]
).abs()

shock_df["Abs_Error_Ensemble"] = (
    shock_df["Actual"]
    - shock_df["ElasticNet_ARIMAX_Pred"]
).abs()

# Đánh dấu năm 2022
shock_df["Period"] = np.where(
    shock_df["MonthYear"].dt.year == 2022,
    "Năm 2022",
    "2023–2024"
)

shock_summary = (
    shock_df
    .groupby("Period")
    .agg(
        ElasticNet_MAE=(
            "Abs_Error_ElasticNet",
            "mean"
        ),
        ElasticNet_ARIMAX_MAE=(
            "Abs_Error_Ensemble",
            "mean"
        ),
        So_thang=(
            "MonthYear",
            "count"
        )
    )
)

shock_summary.round(4)

,ElasticNet_MAE,ElasticNet_ARIMAX_MAE,So_thang
Period,,,
2023–2024,1.6752,1.6911,24
Năm 2022,2.4758,2.4535,12


### Nhận xét về giai đoạn 2022

Kết quả cho thấy sai số dự báo trong năm 2022 cao hơn rõ rệt so với giai đoạn 2023–2024. MAE của ElasticNet tăng từ **1.6752** lên **2.4758**, trong khi ElasticNet + ARIMAX tăng từ **1.6911** lên **2.4535**.

Điều này cho thấy các mô hình gặp nhiều khó khăn hơn khi dự báo trong năm 2022, là giai đoạn xuất hiện nhiều biến động bất thường của thị trường năng lượng. Kết quả này phù hợp với nhận định rằng các cú sốc ngoài mẫu có thể làm giảm khả năng dự báo của mô hình.


## 11. Lưu kết quả dự báo

Kết quả dự báo của các mô hình trên tập Test 2022–2024 được lưu lại để phục vụ bước đánh giá và so sánh mô hình.

Các chỉ số đánh giá như RMSE, MAE, MAPE, R² và DA sẽ được tính trong notebook đánh giá mô hình.

In [ ]:
print("Kích thước:", test_predictions.shape)
print("\nGiá trị thiếu:")
print(test_predictions.isna().sum())

test_predictions.head().round(2)

Kích thước: (36, 8)

Giá trị thiếu:
MonthYear                 0
Actual                    0
Naive_Pred                0
ElasticNet_Pred           0
ARIMAX_Pred               0
Naive_ElasticNet_Pred     0
Naive_ARIMAX_Pred         0
ElasticNet_ARIMAX_Pred    0
dtype: int64


,MonthYear,Actual,Naive_Pred,ElasticNet_Pred,ARIMAX_Pred,Naive_ElasticNet_Pred,Naive_ARIMAX_Pred,ElasticNet_ARIMAX_Pred
0,2022-01,1.18,-1.71,-1.45,-1.56,-1.56,-1.62,-1.51
1,2022-02,2.35,1.18,2.13,1.35,1.74,1.28,1.75
2,2022-03,4.80,2.35,1.95,1.45,2.11,1.83,1.70
3,2022-04,-0.59,4.80,3.66,2.73,4.13,3.61,3.21
4,2022-05,2.34,-0.59,-1.39,-2.35,-1.06,-1.60,-1.86


In [ ]:
test_predictions.to_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/processed/model_predictions.csv",
    index=False
)